# 🔍 Template: Model Interpretability (SHAP + Permutation Importance)

**Purpose:** Explain your champion model — globally and for individual predictions.

**Methods:**
1. Native feature importance (for tree models: gain + split count)
2. Permutation importance (model-agnostic, AUC-based)
3. SHAP global bar plot
4. SHAP beeswarm plot
5. SHAP waterfall plots (individual predictions)
6. Partial Dependence Plots (PDPs)
7. Interpretability triangulation table

**Assumes:** `CHAMPION_NAME`, `CHAMPION_PIPELINE`, `X_train`, `X_val`, `X_test`, `y_test` from previous notebooks.


In [ ]:
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

# ── Extract estimator and scaler from pipeline ────────────────────────────────
champion_estimator = CHAMPION_PIPELINE.named_steps['model']
scaler             = CHAMPION_PIPELINE.named_steps['scaler']

# Transform X sets (needed for SHAP which requires the bare estimator)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test),  columns=X_test.columns,  index=X_test.index)
X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns, index=X_train.index)

# ── SHAP sample: 1,000 rows for tractable compute ────────────────────────────
SHAP_SAMPLE = 1000   # CHANGE: increase for more stable estimates, decrease if slow
rng         = np.random.default_rng(42)
sample_idx  = rng.choice(len(X_test_scaled), size=min(SHAP_SAMPLE, len(X_test_scaled)), replace=False)
X_shap      = X_test_scaled.iloc[sample_idx].reset_index(drop=True)
y_shap      = y_test.iloc[sample_idx].reset_index(drop=True)

print(f'Champion model  : {CHAMPION_NAME}')
print(f'Features        : {X_test.shape[1]}')
print(f'SHAP sample     : {len(X_shap):,} rows')
print(f'Feature names   : {list(X_test.columns)}')


## 1️⃣  Native Feature Importance (Tree Models Only)

Skip this section for Logistic Regression or non-tree models.

In [ ]:
# ── Works for LightGBM, XGBoost, RandomForest, GradientBoosting ─────────────
try:
    # LightGBM
    if hasattr(champion_estimator, 'booster_'):
        imp_gain  = champion_estimator.booster_.feature_importance(importance_type='gain')
        imp_split = champion_estimator.booster_.feature_importance(importance_type='split')
        importance_df = pd.DataFrame({
            'gain' : imp_gain,
            'split': imp_split,
        }, index=X_test.columns)
    # XGBoost / sklearn tree models
    elif hasattr(champion_estimator, 'feature_importances_'):
        importance_df = pd.DataFrame({
            'gain': champion_estimator.feature_importances_,
        }, index=X_test.columns)
    else:
        raise AttributeError('No native importance available for this model type.')

    importance_df['gain_pct'] = importance_df['gain'] / importance_df['gain'].sum() * 100
    importance_df = importance_df.sort_values('gain', ascending=False)
    print('Feature Importance (top 15):')
    print(importance_df.head(15).round(4).to_string())

    top_n = min(15, len(importance_df))
    fig, ax = plt.subplots(figsize=(10, 7), dpi=100)
    top = importance_df['gain'].head(top_n).sort_values()
    ax.barh(top.index, top.values, color='#1a434e', edgecolor='white')
    ax.set(xlabel='Gain', title=f'Feature Importance (Gain) — {CHAMPION_NAME}')
    sns.despine(); plt.tight_layout(); plt.show()

except AttributeError as e:
    print(f'Native importance not available: {e}')


## 2️⃣  Permutation Importance (Model-Agnostic)

In [ ]:
# Computed on the full pipeline (scaling applied inside each repeat)
perm_result = permutation_importance(
    CHAMPION_PIPELINE, X_val, y_val,
    n_repeats    = 10,
    scoring      = 'roc_auc',
    random_state = 42,
    n_jobs       = -1,
)
perm_df = pd.DataFrame({
    'feature'  : X_val.columns,
    'mean_drop': perm_result.importances_mean,
    'std_drop' : perm_result.importances_std,
}).sort_values('mean_drop', ascending=False).reset_index(drop=True)

print('Permutation Importance (AUC drop, top 15):')
print(perm_df.head(15).to_string(index=False, float_format='{:.5f}'.format))

fig, ax = plt.subplots(figsize=(10, 7), dpi=100)
top15 = perm_df.head(15).sort_values('mean_drop')
colors = ['#e74c3c' if v < 0 else '#1a434e' for v in top15['mean_drop']]
ax.barh(top15['feature'], top15['mean_drop'],
        xerr=top15['std_drop'], color=colors,
        error_kw={'ecolor':'#555','capsize':4}, edgecolor='white')
ax.axvline(x=0, color='black', lw=0.8)
ax.set(xlabel='Mean AUC-ROC Drop', title='Permutation Importance — Validation Set')
sns.despine(); plt.tight_layout(); plt.show()


## 3️⃣  SHAP — Global Feature Importance

In [ ]:
# TreeExplainer is fast and exact for tree-based models.
# Use KernelExplainer for linear/non-tree models (slower).
try:
    explainer   = shap.TreeExplainer(champion_estimator)
except Exception:
    # Fallback: KernelExplainer (slower, works for any model)
    explainer   = shap.KernelExplainer(
        lambda x: champion_estimator.predict_proba(x)[:, 1],
        shap.sample(X_shap, 100)
    )

shap_values = explainer(X_shap)
print(f'SHAP values shape : {shap_values.values.shape}')

# Global bar chart
fig, ax = plt.subplots(figsize=(10, 7), dpi=100)
shap.plots.bar(shap_values, max_display=20, show=False, ax=ax)
ax.set_title(f'SHAP — Mean |SHAP| Feature Importance\n{CHAMPION_NAME}', fontweight='bold', loc='left')
plt.tight_layout(); plt.show()

# Mean absolute SHAP values for ranking
mean_abs_shap = np.abs(shap_values.values).mean(axis=0)


## 4️⃣  SHAP — Beeswarm Plot

In [ ]:
plt.figure(figsize=(11, 9), dpi=100)
shap.plots.beeswarm(shap_values, max_display=20, show=False)
plt.title(f'SHAP Beeswarm — Feature Impact Distribution\n{CHAMPION_NAME}', fontweight='bold', loc='left')
plt.tight_layout(); plt.show()


## 5️⃣  SHAP — Individual Waterfall Plots

In [ ]:
# Identify high-risk, low-risk, and borderline cases
shap_proba      = champion_estimator.predict_proba(X_shap)[:, 1]

# CHANGE: adjust threshold to your optimal threshold from evaluation notebook
OPTIMAL_THRESHOLD = 0.5

high_risk_idx  = int(np.argmax(shap_proba))
low_risk_idx   = int(np.argmin(shap_proba))
border_idx     = int(np.argmin(np.abs(shap_proba - OPTIMAL_THRESHOLD)))

cases = {
    'High-Risk'  : high_risk_idx,
    'Low-Risk'   : low_risk_idx,
    'Borderline' : border_idx,
}

for label, idx in cases.items():
    prob   = shap_proba[idx]
    actual = 'Positive' if y_shap.iloc[idx] == 1 else 'Negative'
    print(f'{label}: P(positive) = {prob:.4f}  |  Actual = {actual}')
    plt.figure(figsize=(12, 5), dpi=100)
    shap.plots.waterfall(shap_values[idx], max_display=12, show=False)
    plt.title(f'SHAP Waterfall — {label}\nP(positive) = {prob:.4f}  |  Actual: {actual}',
              fontweight='bold', loc='left')
    plt.tight_layout(); plt.show()
    print()


## 6️⃣  Partial Dependence Plots (top 4 features)

In [ ]:
top4_idx      = np.argsort(mean_abs_shap)[::-1][:4]
top4_features = [X_shap.columns[i] for i in top4_idx]
print(f'Top 4 features by |SHAP|: {top4_features}')

fig, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=100)
PartialDependenceDisplay.from_estimator(
    CHAMPION_PIPELINE, X_val, features=top4_features,
    kind='average', grid_resolution=50, n_jobs=-1,
    ax=axes.flatten(), line_kw={'color': '#1a434e', 'lw': 2.5},
)
for ax, feat in zip(axes.flatten(), top4_features):
    ax.set_title(f'PDP — {feat}', fontweight='bold', loc='left')
    sns.despine(ax=ax)
plt.suptitle('Partial Dependence Plots — Top 4 Features by SHAP', fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()


## 7️⃣  Interpretability Triangulation Table

In [ ]:
# Rank each feature across all three methods
try:
    gain_rank = importance_df['gain'].rank(ascending=False).astype(int)
except NameError:
    gain_rank = pd.Series(dtype=int)

perm_rank  = perm_df.set_index('feature')['mean_drop'].rank(ascending=False).astype(int)
shap_rank  = pd.Series(mean_abs_shap, index=X_shap.columns).rank(ascending=False).astype(int)

summary_df = pd.DataFrame({'SHAP Rank': shap_rank, 'Permutation Rank': perm_rank})
if not gain_rank.empty:
    summary_df['Gain Rank'] = gain_rank
summary_df = summary_df.dropna()
summary_df['Avg Rank'] = summary_df.mean(axis=1)
summary_df = summary_df.sort_values('Avg Rank')
summary_df.index.name = 'Feature'

print('='*60)
print('INTERPRETABILITY TRIANGULATION — Feature Rank by Method')
print('='*60)
print(summary_df.head(15).round(1).to_string())
